# 01 · Silence Anatomy — how much quiet is in MUSDB vocals, and why the field can't measure the leak

**How much of a vocal stem is truly silent, what does the silence-leakage metric (SLR)
actually measure, and how do four chunk-sampling policies reweight the silence a model sees
during training?** This notebook is the visual companion to [`../THEORY.md`](../THEORY.md):
it tells the measurement-gap story (museval NaNs silent frames; SI-SDR is singular on a
silent target), walks the silent-region identifier on constructed audio, demonstrates the
SLR anchors numerically (do-nothing → 0 dB, perfect → ε floor, 10 % leak → −10 dB), and
draws the four policies' sampling-weight profiles over one track.

Nothing here needs a GPU. Most cells run on CPU from synthetic signals + the `singnet`
functions alone; the two that need a decoded MUSDB shard (the real activity-profile figures)
are marked **⚠️ RUN THIS LATER**. The notebook ships **un-executed** so the committed file is
a clean scaffold.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (the SLR metric), §4.1 (the four
  policies); [`../THEORY.md`](../THEORY.md) §2 (the measurement gap — the crux), §3 (SLR
  properties — the crux), §4 (policies as exposure distributions).
- **Data prep is *not* repeated here.** Acquisition, licensing, the 86/14/50 split, and the
  STFT front end live in Direction 01's notebooks
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)).
  This direction reuses them and adds only the energy-profile pass (§5).
- **Code, not prose, is authoritative:** the metric is `singnet.metrics.slr`; the policies
  are `singnet.data.sampling`; the profiles are `singnet.data.profiles`. Every number below
  is asserted in `tests/` (gate G0).

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · The measurement gap: why SDR can't see ghost vocals (THEORY §2)

The karaoke-critical failure is a **ghost vocal** — energy leaking into the vocal estimate
during an instrumental break. The field-standard metrics are structurally blind to it:
**museval** sets every silent-reference frame to `NaN` and drops it from the median (verified
from `metrics.py`, `../../00-shared-research/papers/bsseval-museval-sisec2018.md` §3), and
**SI-SDR** divides by ‖v‖² = 0 on a silent target. The cell shows both pathologies on a
constructed silent-target chunk, and that **SLR is finite** there because it references the
loud *mixture*, not the silent vocal.

In [ ]:
# CPU-runnable now: SI-SDR is undefined on a silent target; SLR is finite (THEORY §2-3).
import numpy as np
from singnet.metrics import si_sdr, slr

sr = 16000
vocal_gt = np.zeros(sr, dtype=np.float64)                        # v = 0  (instrumental break)
mixture = 0.5 * np.random.default_rng(0).standard_normal(sr)     # x = a  (loud accompaniment)
leak = np.sqrt(0.1) * mixture                                    # the model leaks 10% of a's ENERGY

# SI-SDR against a silent target: alpha = <est,v>/||v||^2 -> divide by zero.
print(f"||v||^2 = {float(vocal_gt @ vocal_gt):.1f}  -> SI-SDR alpha divides by zero (undefined)")
print("si_sdr(leak, v=0, eps=0) =", si_sdr(leak, vocal_gt, eps=0.0), "(singular / -inf)")

# museval would NaN this whole frame (ref source silent) and drop it from the median.
print("museval: frame with silent reference -> NaN, dropped from the median (THEORY 2.2)")

# SLR references the MIXTURE, so it is finite and reads the leak directly.
print("SLR(leak, mix) =", round(slr(leak, mixture, [(0, sr)]), 3), "dB   (10% energy -> -10 dB)")

## 2 · The silent-region identifier (THEORY §2.1)

`silent_regions(vocal_wave, sr, theta_db, frame_s, hop_s, min_run_s)` frames the GT vocal
(100 ms / 50 ms hop), flags frames below θ dBFS, merges consecutive silent frames into runs,
and keeps runs ≥ `L_min = 0.5 s`. The cell constructs a loud-then-quiet-then-loud vocal and
shows (a) the kept silent run, (b) that a short 0.3 s gap is filtered, and (c) that θ moves
the set (the §2.1 / §3.6 sensitivity axis).

In [ ]:
# CPU-runnable now: the silent-region identifier on constructed audio (θ, L_min, runs).
import numpy as np
import matplotlib.pyplot as plt
from singnet.metrics import silent_regions

sr = 1000
loud = np.full(int(1.0 * sr), 0.5)                       # -6 dBFS, clearly active
brk = np.zeros(int(1.2 * sr))                            # a 1.2 s instrumental break
short_gap = np.zeros(int(0.3 * sr))                      # a 0.3 s breath (< L_min)
vocal = np.concatenate([loud, brk, loud, short_gap, loud]).astype(np.float64)

regions = silent_regions(vocal, sr, theta_db=-60.0, min_run_s=0.5)
print("kept silent regions (samples):", regions, "-> the 1.2 s break only (0.3 s gap filtered)")

fig, ax = plt.subplots(figsize=(9, 2.6))
t = np.arange(vocal.size) / sr
ax.plot(t, vocal, lw=0.8, color="#2b6cb0", label="GT vocal")
for s, e in regions:
    ax.axvspan(s / sr, e / sr, color="#c0392b", alpha=0.25)
ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude")
ax.set_title("silent_regions: kept runs >= L_min = 0.5 s (theta = -60 dBFS)")
fig.tight_layout(); plt.show()

# theta sensitivity: a quieter (-65 dBFS) passage counts as silent under -60 but not -70.
quiet = np.full(int(1.0 * sr), 10.0 ** (-65.0 / 20.0))
probe = np.concatenate([loud, quiet]).astype(np.float64)
print("theta=-60 -> n regions:", len(silent_regions(probe, sr, theta_db=-60.0)),
      "| theta=-70 -> n regions:", len(silent_regions(probe, sr, theta_db=-70.0)))

In [ ]:
# RUN THIS LATER (CPU, needs one decoded MUSDB shard) — real vocal-activity anatomy.
# For real MUSDB tracks: the GT-vocal waveform with its R_sil regions shaded, and the
# per-track silent-fraction phi (fraction of the 1-s-grid 6-s windows below theta) across the
# 14 val tracks -- the section-1.1 statistic every policy reweights. Also the valid-n bar (how
# many of the 14 val tracks have >= 1 silent run >= L_min at each theta -- the G2 gate).
#
#   from singnet.data import WavShardStore, load_manifest
#   from singnet.metrics import silent_regions
#   store = WavShardStore(os.environ["SHARD_ROOT"]); manifest = load_manifest("01-.../splits.csv")
#   for track in manifest.tracks_for("valid"):
#       v = store.load_sources(track)["vocals"]
#       regions = silent_regions(v, store.sample_rate, theta_db=-60.0)
#       ...  # shade regions; accumulate silent-fraction + valid-n at theta in {-50,-60,-70}
print("Real MUSDB activity anatomy -- RUN LATER (needs a decoded shard).")

## 3 · SLR anchors, demonstrated numerically (THEORY §3.2)

The anchors are what make SLR readable. On one constructed silent region: a **do-nothing**
separator (v̂ = x) scores **exactly 0 dB**; a **perfect** one (v̂ = 0) scores at the **ε
floor**; a **10 %-energy** leak scores **−10 dB**; and SLR is **joint-gain invariant**
(scaling est and mix together leaves it unchanged).

In [ ]:
# CPU-runnable now: the three SLR anchors + joint-gain invariance (all asserted in tests/).
import numpy as np
from singnet.metrics import slr
from singnet.metrics.slr import DEFAULT_EPS

rng = np.random.default_rng(1)
mix = rng.standard_normal(8192)
region = [(0, 8192)]
mix_energy = float(np.sum(mix ** 2))

print("do-nothing  (v=x)        SLR =", slr(mix, mix, region), "dB   (exactly 0)")
print("perfect     (v=0)        SLR =", round(slr(np.zeros_like(mix), mix, region), 2),
      "dB   (eps floor =", round(10 * np.log10(DEFAULT_EPS / (mix_energy + DEFAULT_EPS)), 2), "dB)")
print("10% energy  (sqrt.1 * x) SLR =", round(slr(np.sqrt(0.1) * mix, mix, region), 4), "dB   (-10)")
print("1%  energy  (sqrt.01* x) SLR =", round(slr(np.sqrt(0.01) * mix, mix, region), 4), "dB   (-20)")

est = 0.3 * mix + 0.05 * rng.standard_normal(8192)
base = slr(est, mix, region, eps=0.0)
scaled = slr(10.0 * est, 10.0 * mix, region, eps=0.0)   # joint gain x10
print(f"joint-gain invariance: SLR(est,mix)={base:.4f}  SLR(10est,10mix)={scaled:.4f}  (equal)")

## 4 · The four policies' sampling-weight profiles over one track (THEORY §4)

Given a track's windowed vocal-energy profile `E(s)` (the §5 prep artifact), each policy
induces a chunk-start distribution (`ChunkSampler.weights`). On a constructed profile with a
silent middle, the cell draws all four: **uniform** (flat), **energy** (∝ E with a λ floor),
**drop** (uniform on the supported starts, zero on silent ones), and **curriculum** at t=0
(uniform-like) — and prints the λ(t) anneal.

In [ ]:
# CPU-runnable now: the four policies' chunk-start weights over one synthetic profile.
import numpy as np
import matplotlib.pyplot as plt
from singnet.data import ChunkSampler

# a track profile: active, then a truly silent middle (< -60 dBFS = 0.001), then active
# again (1-s grid). Starts 3-5 sit below the drop threshold, so drop excludes them.
E = np.array([0.30, 0.35, 0.28, 0.0002, 0.0001, 0.0002, 0.31, 0.29, 0.33, 0.30])
N = E.size
total_steps = 16000

w_uniform = ChunkSampler("uniform").weights(E)
w_energy = ChunkSampler("energy", floor_lambda=0.1).weights(E)
w_drop = ChunkSampler("drop", theta_db=-60.0).weights(E)
curr = ChunkSampler("curriculum", floor_lambda=0.1, total_steps=total_steps)

fig, ax = plt.subplots(figsize=(9, 3.4))
g = np.arange(N)
ax.bar(g - 0.3, w_uniform, width=0.2, label="uniform", color="#8899aa")
ax.bar(g - 0.1, w_energy, width=0.2, label="energy (lam=0.1)", color="#2b6cb0")
ax.bar(g + 0.1, w_drop, width=0.2, label="drop (theta=-60)", color="#c0392b")
ax.bar(g + 0.3, curr.weights(E, step=0), width=0.2, label="curriculum t=0 (lam=1)", color="#e0b030")
ax.plot(g, E / E.sum(), "k--", lw=1, alpha=0.6, label="E(s)/sum E (activity)")
ax.set_xlabel("chunk-start grid index"); ax.set_ylabel("draw probability")
ax.set_title("Chunk-start distributions by policy (silent middle = starts 3-5)")
ax.legend(fontsize=8, ncol=2); fig.tight_layout(); plt.show()

# drop puts ZERO mass on the silent starts; energy keeps them >= lam/N (never starved).
print("drop weights on silent starts 3-5 :", np.round(w_drop[3:6], 4), "(excluded)")
print("energy weights on silent starts 3-5:", np.round(w_energy[3:6], 4),
      "(>= lam/N =", round(0.1 / N, 4), ")")
for frac in (0.0, 0.25, 0.5, 1.0):
    print(f"curriculum lam(t={int(frac*total_steps):>5}) = {curr.lambda_at(int(frac*total_steps)):.3f}")

In [ ]:
# RUN THIS LATER (CPU, needs energy_profiles.json from --write-energy-profiles) --
# the four policies' weights over a REAL MUSDB track's vocal-activity profile.
#
#   from singnet.data import load_energy_profiles, ChunkSampler
#   profiles = load_energy_profiles(os.path.join(os.environ["SHARD_ROOT"], "energy_profiles.json"))
#   E = profiles["<a real train track>"]
#   # same four-bar plot as above, now over the real activity curve; report each policy's
#   # expected silent exposure sum_{s in S} w(s) vs the closed forms (uniform=phi, drop=0, energy~lam*phi).
print("Real-track policy weights -- RUN LATER (needs the energy-profile prep pass).")

## 5 · Takeaways

- **The gap is real and code-verified:** museval NaNs silent frames, SI-SDR is singular on a
  silent target — so no standard metric ranks two systems by ghost-vocal behavior. SLR,
  computed only on `R_sil`, is the complementary axis (THEORY §2).
- **SLR is readable by construction:** do-nothing = 0 dB, perfect = ε floor, 10 % leak =
  −10 dB, and it is joint-gain invariant — anchors asserted to numerical precision (§3).
- **The policies reweight silence exactly:** uniform sees it at the base rate φ, drop at 0,
  energy at ≈ λφ (down-weighted, never starved), curriculum front-loaded (§4–§5). Notebook 02
  turns these exposure differences into the (SI-SDR, SLR) tradeoff plane.